In [ ]:
# NOTEBOOK PARAMETERS
selected_model = 'neptune_1'
ensemble_size = 8
years = [2019,2020,2021]

# Ocean Heat Content

Formula taken from 
https://gallery.pangeo.io/repos/NCAR/notebook-gallery/notebooks/Run-Anywhere/Ocean-Heat-Content/OHC_tutorial.html

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import xarray as xr
import numpy as np
import os
import sys

sys.path.append('library')
from atmocean.visualize import (
    get_colormap_from_cmap,
    get_sst_bias_colormap,
    plot_dmi_timeseries,
    plot_dmi_timeseries_optim,
    plot_map_on_grid,
    discrete_cmap,
)
from atmocean.enso import (
    monthly_resample_with_prediction_timedelta,
    monthly_groupby_with_prediction_timedelta,
    # compute_ocean_heat,
    compute_ocean_heat_optim,
)
from atmocean.utils import (
    clean_forecasts,
    remove_ocean_variables_suffix,
    # align_prediction_timedelta,
    align_prediction_timedelta_optim,
)
os.system('hostname')
%load_ext autoreload
%autoreload 2

In [ ]:
# get the forecasting horizon for this run
start_date = '01-01'
end_date = '12-31'
lead_times = [0, 10, 20, 30, 40, 50, 60]
max_depth = 300

main_dir = 'PATH-TO-MAIN-DIRECTORY'

data_dir = os.path.join(main_dir, 'data')
forecast_dir = os.path.join(data_dir, 'forecasts')

if ensemble_size is None:
    forecasts_srcs = [
        os.path.join(forecast_dir, f'{selected_model}-from_{year}-{start_date}_to_{year}-{end_date}.zarr')
        for year in years
    ]
    out_img_dir = os.path.join(main_dir, f'validation/img/{selected_model}/s2s-{years[0]}-{years[-1]}/physical_coherency/ohc')
else:
    forecasts_srcs = [
        os.path.join(forecast_dir, f'{selected_model}-from_{year}-{start_date}_to_{year}-{end_date}-ENS-{ensemble_size:02d}.zarr')
        for year in years
    ]
    out_img_dir = os.path.join(main_dir, f'validation/img/{selected_model}/s2s-{years[0]}-{years[-1]}-ENS-{ensemble_size:02d}/physical_coherency/ohc')

os.makedirs(out_img_dir, exist_ok=True)
plot_kwargs = dict(dpi=100, bbox_inches='tight')
save_figures = True

In [ ]:
from dask.distributed import Client, LocalCluster, progress, wait
from tqdm import tqdm
import numpy as np
import dask
import ctypes

# spill to disk before hitting hard memory limits (avoids KilledWorker)
dask.config.set({
    'distributed.worker.memory.target': 0.6,
    'distributed.worker.memory.spill': 0.7,
    'distributed.worker.memory.pause': 0.8,
    'distributed.worker.memory.terminate': 0.95,
})

cpu_count = os.cpu_count() or 16

n_workers = min(16, cpu_count // 2)
threads_per_worker = 2  # o 4

memory_limit = 948 // n_workers

print(f'memory_limit {memory_limit}GB')
print(f'n_workers {n_workers}')
print(f'threads_per_worker {threads_per_worker}')
cluster = LocalCluster(
    n_workers=n_workers, threads_per_worker=threads_per_worker,
    memory_limit=f'{memory_limit}GB',
    local_directory=os.path.join(main_dir, 'tmp_dask'),  # spill directory
)
client = Client(cluster)

def _trim_memory() -> int:
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

client.run(_trim_memory)
client

In [ ]:
# open forecasts
target_mib = 128
dask.config.set({'array.chunk-size': f'{target_mib}MiB'})

chunks = {'time': 5, 'prediction_timedelta': 1, 'latitude': -1, 'longitude': -1}
if ensemble_size is not None:
    chunks['ensemble_member'] = 1

forecasts = xr.open_mfdataset(
    forecasts_srcs, chunks={}, parallel=True, engine='zarr', preprocess=clean_forecasts,
    combine='nested', concat_dim='time', data_vars='minimal', coords='minimal', compat='override',
)

# manipulate the time dimension to align
forecasts = forecasts.isel(prediction_timedelta=lead_times, level=0, missing_dims='ignore')
# remove ocean variables suffix (oras, cglo, glor)
forecasts = remove_ocean_variables_suffix(forecasts)

In [ ]:
# Ocean Heat Content is computed only up to 300 meters of depth
forecasts = forecasts.sel(depth=slice(max_depth)).chunk({'depth': -1})

In [ ]:
delta_level = forecasts.depth.values[1:] - forecasts.depth.values[:-1]
delta_level = xr.Dataset(coords=dict(depth=delta_level))

In [ ]:
forecasts['depth'] = np.r_[delta_level.depth.values, np.nan]
forecasts = forecasts.isel(depth=slice(0,-1))

In [ ]:
forecasts_heat = compute_ocean_heat_optim(delta_level, forecasts[f'thetao'])
forecasts_heat = client.persist(forecasts_heat)
progress(forecasts_heat)
client.run(_trim_memory)

In [ ]:
coords = dict(
    time=forecasts.time,
    prediction_timedelta=forecasts.prediction_timedelta,
    latitude=forecasts.latitude,
    longitude=forecasts.longitude,
)
forecasts_heat_ds = xr.Dataset(coords=coords).astype(np.float32)

if ensemble_size is None: fhcoords = ('time', 'prediction_timedelta', 'latitude', 'longitude')
else: fhcoords = ('ensemble_member', 'time', 'prediction_timedelta', 'latitude', 'longitude')

forecasts_heat_ds['heat'] = (fhcoords, forecasts_heat.data)

In [ ]:
dates = pd.to_datetime(forecasts.time.values)
start_time, end_time = dates[0]+pd.DateOffset(days=lead_times[-1]), dates[-1]
start_time, end_time

In [ ]:
forecasts_heat_ds = align_prediction_timedelta_optim(forecasts_heat_ds)

In [ ]:
if ensemble_size is None:
    forecasts_heat_timeseries_ds = forecasts_heat_ds.mean(dim=['latitude','longitude'], skipna=True)
else:
    forecasts_heat_timeseries_ds = forecasts_heat_ds.mean(dim=['ensemble_member','latitude','longitude'], skipna=True)
forecasts_heat_timeseries_ds = client.persist(forecasts_heat_timeseries_ds)
progress(forecasts_heat_timeseries_ds)

In [ ]:
colors = discrete_cmap(len(lead_times)+2, 'Blues_r')
colors = [colors(i) for i in range(len(lead_times))]

## OHC Timeseries

In [ ]:
models_colors = {
    f'{lead_time}': colors[i]
    for i, lead_time in enumerate(lead_times)
}
models_names = {
    f'{lead_time}': 'Ground Truth' if lead_time == 0 else f"t + {lead_time}"
    for lead_time in lead_times
}
models_data = {}
for i,lt in enumerate(lead_times):
    models_data[f'{lt}'] = forecasts_heat_timeseries_ds['heat'].isel(prediction_timedelta=i)

In [ ]:
from scipy.stats import spearmanr

outfile = os.path.join(out_img_dir, 'ocean_heat_content_timeseries.png') if save_figures else None
plot_dmi_timeseries_optim(
    data=models_data,
    colors=models_colors,
    names=models_names,
    dmi=None,
    threshold=None,
    dmi_label=False,
    outfile=outfile,
)

In [ ]:
heat = forecasts_heat_ds['heat']
if ensemble_size is None:
    heat = heat.mean(dim='time', skipna=True).compute()
else:
    heat = heat.mean(dim=['time','ensemble_member'], skipna=True).compute()

## OHC Spatial Anomaly

In [ ]:
from matplotlib.ticker import ScalarFormatter
from matplotlib.gridspec import GridSpec
import matplotlib.patches as mpatches
from matplotlib import pyplot as plt
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.crs as ccrs

In [ ]:
grid_list = [
    'gs[0,0:2]',
    'gs[0,2:4]',
    'gs[0,4:6]',
    'gs[1,0:2]',
    'gs[1,2:4]',
    'gs[1,4:6]',
    'gs[2,2:4]',
]

In [ ]:
cmap, norm, levels = get_sst_bias_colormap(2.0e11, 2.6e11)

heat_list = []
for i,lt in enumerate(lead_times):
    heat_list.append(
        heat.isel(prediction_timedelta=i)
    )

central_longitude = 180
lats, lons = heat_list[0].latitude.data, heat_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(10,5.5))
fig.suptitle(f'Ocean Heat Content', fontsize=16, y=1.05)

axes.axis('off')

# initialize grid spec
gs = GridSpec(3, 6, figure=fig)
axs = []

for i,lt in enumerate(lead_times):
    im, ax = plot_map_on_grid(
        bias=heat_list[i], grid=eval(grid_list[i]), title='Ground Truth' if i == 0 else f't+{lt}',
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, 
        cmap=cmap, norm=norm, levels=levels, use_lat=False, use_lon=False
    )
    axs.append(ax)

# single colorbar
cax = fig.add_axes([0.25, -0.05, 0.5, 0.03]) # [left, bottom, width, height]
cbar = fig.colorbar(
    im, ax=axs, cax=cax, 
    orientation='horizontal', pad=0.05, 
    aspect=60, shrink=0.9, label='J/K', 
    format=ScalarFormatter(), 
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=-0.3, hspace=0.5
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'ocean_heat_content_spatial.png')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()
plt.close(fig)
# heat_list[-1].min(skipna=True), heat_list[-1].max(skipna=True)

In [ ]:
cmap_, norm_, levels_ = get_colormap_from_cmap('RdBu_r', -3.5e8, 3.5e8)

heat_bias_list = []
for i,lt in enumerate(lead_times[1:]):
    heat_bias_list.append(
        heat.isel(prediction_timedelta=i+1) - heat.isel(prediction_timedelta=0)
    )

central_longitude = 180
lats, lons = heat_bias_list[0].latitude.data, heat_bias_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(16,5.5))
fig.suptitle(f'Ocean Heat Content Bias', fontsize=16, y=1.05)

axes.axis('off')

# initialize grid spec
gs = GridSpec(2, 6, figure=fig)
axs = []

for i,lt in enumerate(lead_times[1:]):
    im, ax = plot_map_on_grid(
        bias=heat_bias_list[i], grid=eval(grid_list[i]), title=f't+{lt}',
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, 
        cmap=cmap_, norm=norm_, levels=levels_, use_lat=False, use_lon=False
    )
    axs.append(ax)

# single colorbar
cax = fig.add_axes([0.25, -0.05, 0.5, 0.03]) # [left, bottom, width, height]
cbar = fig.colorbar(
    im, ax=axs, cax=cax, 
    orientation='horizontal', pad=0.05, 
    aspect=60, shrink=0.9, label='J/K', 
    format=ScalarFormatter(), 
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=-0.1, hspace=0.3
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'ocean_heat_content_bias_spatial.png')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()
plt.close(fig)
# heat_bias_list[-1].min(skipna=True), heat_bias_list[-1].max(skipna=True)

In [ ]:
cmap_, norm_, levels_ = get_colormap_from_cmap('RdBu_r', -3.5e8, 3.5e8)

heat_bias_list = []
for i,lt in enumerate(lead_times[1:]):
    heat_bias_list.append(
        heat.isel(prediction_timedelta=i+1) - heat.isel(prediction_timedelta=i)
    )

central_longitude = 180
lats, lons = heat_bias_list[0].latitude.data, heat_bias_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(16,5.5))
fig.suptitle(f'Ocean Heat Content Bias', fontsize=16, y=1.05)

axes.axis('off')

# initialize grid spec
gs = GridSpec(2, 6, figure=fig)
axs = []

for i,lt in enumerate(lead_times[1:]):
    im, ax = plot_map_on_grid(
        bias=heat_bias_list[i], grid=eval(grid_list[i]), title=f't+{lt}',
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, 
        cmap=cmap_, norm=norm_, levels=levels_, use_lat=False, use_lon=False
    )
    axs.append(ax)

# single colorbar
cax = fig.add_axes([0.25, -0.05, 0.5, 0.03]) # [left, bottom, width, height]
cbar = fig.colorbar(
    im, ax=axs, cax=cax, 
    orientation='horizontal', pad=0.05, 
    aspect=60, shrink=0.9, label='J/K', 
    format=ScalarFormatter(), 
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=-0.1, hspace=0.3
)

# save or plot figure
# outfile = os.path.join(out_img_dir, f'ocean_heat_content_bias_spatial.png')
# plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()
# plt.close(fig)

In [ ]:
lead_times_figure = [0, 10, 30, 60]
grid_list_top = [
    'gs[0,0:2]',
    'gs[0,2:4]',
    'gs[0,4:6]',
    'gs[0,6:8]',
]
grid_list_bottom = [
    'gs[1,0:2]',
    'gs[1,2:4]',
    'gs[1,4:6]',
    'gs[1,6:8]',
]

cmap_top, norm_top, levels_top = get_sst_bias_colormap(2.0e11, 2.6e11)
cmap_bottom, norm_bottom, levels_bottom = get_colormap_from_cmap('RdBu_r', -3.5e8, 3.5e8)

ohc_list = []
ohc_bias_list = []
for i,lt in enumerate(lead_times_figure):
    ohc_list.append(
        heat.isel(prediction_timedelta=lt//10)
    )
    ohc_bias_list.append(
        heat.isel(prediction_timedelta=lt//10) - heat.isel(prediction_timedelta=0)
    )

central_longitude = 180
lats, lons = ohc_list[0].latitude.data, ohc_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(12,3))

axes.axis('off')

# initialize grid spec
gs = GridSpec(2, 8, figure=fig)
axs_top, axs_bottom = [], []
eps = 0.0 # 1e-2

for i,lt in enumerate(lead_times_figure):
    im_top, ax = plot_map_on_grid(
        bias=ohc_list[i], grid=eval(grid_list_top[i]), title='Ground Truth' if i == 0 else f't+{lt}', title_kwargs=dict(fontsize=11), 
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, cmap=cmap_top, norm=norm_top, levels=levels_top
    )
    axs_top.append(ax)

for i,lt in enumerate(lead_times_figure):
    if i == 0: 
        ax = fig.add_subplot(eval(grid_list_bottom[0]))
        ax.axis('off')
        continue
    im_bottom, ax = plot_map_on_grid(
        bias=ohc_bias_list[i], grid=eval(grid_list_bottom[i]), title='', title_kwargs=dict(fontsize=11), # ='Ground Truth' if i == 0 else f't+{lt}',
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, cmap=cmap_bottom, norm=norm_bottom, levels=levels_bottom
    )
    axs_bottom.append(ax)

# single colorbar
cax_top = fig.add_axes([0.95, 0.55, 0.01, 0.4]) # [left, bottom, width, height]
cax_bottom = fig.add_axes([0.95, 0.05, 0.01, 0.4]) # [left, bottom, width, height]
cbar_top = fig.colorbar(
    im_top, ax=axs_top, cax=cax_top, orientation='vertical', label='J/K',
)
cbar_bottom = fig.colorbar(
    im_bottom, ax=axs_bottom, cax=cax_bottom, orientation='vertical', label='J/K',
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=0.0, hspace=0.3
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'ocean_heat_content.pdf')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()
plt.close(fig)

In [ ]:
lead_times_figure = [0, 10, 30, 60]
grid_list_top = [
    'gs[0,0:2]',
    'gs[0,2:4]',
    'gs[0,4:6]',
    'gs[0,6:8]',
]
grid_list_bottom = [
    'gs[1,0:2]',
    'gs[1,2:4]',
    'gs[1,4:6]',
    'gs[1,6:8]',
]

cmap_top, norm_top, levels_top = get_sst_bias_colormap(2.0e11, 2.6e11)
cmap_bottom, norm_bottom, levels_bottom = get_colormap_from_cmap('RdBu_r', -3.5e8, 3.5e8)

ohc_list = []
ohc_bias_list = []
for i,lt in enumerate(lead_times_figure):
    ohc_list.append(
        heat.isel(prediction_timedelta=lt//10)
    )
    ohc_bias_list.append(
        heat.isel(prediction_timedelta=lt//10) - heat.isel(prediction_timedelta=0)
    )

central_longitude = 180
lats, lons = ohc_list[0].latitude.data, ohc_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(12,3))

axes.axis('off')

# initialize grid spec
gs = GridSpec(2, 8, figure=fig)
axs_top, axs_bottom = [], []
eps = 0.0 # 1e-2

for i,lt in enumerate(lead_times_figure):
    im_top, ax = plot_map_on_grid(
        bias=ohc_list[i], grid=eval(grid_list_top[i]), title='Ground Truth' if i == 0 else f't+{lt}', title_kwargs=dict(fontsize=11), 
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, cmap=cmap_top, norm=norm_top, levels=levels_top
    )
    axs_top.append(ax)

for i,lt in enumerate(lead_times_figure):
    if i == 0: 
        ax = fig.add_subplot(eval(grid_list_bottom[0]))
        ax.axis('off')
        continue
    im_bottom, ax = plot_map_on_grid(
        bias=ohc_bias_list[i], grid=eval(grid_list_bottom[i]), title='', title_kwargs=dict(fontsize=11), # ='Ground Truth' if i == 0 else f't+{lt}',
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, cmap=cmap_bottom, norm=norm_bottom, levels=levels_bottom
    )
    axs_bottom.append(ax)

# single colorbar
cax_top = fig.add_axes([0.95, 0.55, 0.01, 0.4]) # [left, bottom, width, height]
cax_bottom = fig.add_axes([0.95, 0.05, 0.01, 0.4]) # [left, bottom, width, height]
cbar_top = fig.colorbar(
    im_top, ax=axs_top, cax=cax_top, orientation='vertical', label='J/K',
)
cbar_bottom = fig.colorbar(
    im_bottom, ax=axs_bottom, cax=cax_bottom, orientation='vertical', label='J/K',
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=0.0, hspace=0.3
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'ocean_heat_content.png')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()
plt.close(fig)

## OHC Hovmoller

### El Nino OHC Hovmoller


- asse y = time
- asse x = longitude 130°E-283°E (average at latitude between 2°S-2°N)

In [ ]:
heat_ds = forecasts_heat_ds['heat']
if ensemble_size is None:
    heat_ds = heat_ds.sel(latitude=slice(2,-2), longitude=slice(130, 280)).mean(dim='latitude', skipna=True).compute()
else:
    heat_ds = heat_ds.sel(latitude=slice(2,-2), longitude=slice(130, 280)).mean(dim=['ensemble_member','latitude'], skipna=True).compute()

In [ ]:
cmap2, norm2, levels2 = get_sst_bias_colormap(2.3e11, 2.5e11)
grid_list_ = [
    'gs[0:2,0]',
    'gs[0:2,1]',
    'gs[0:2,2]',
    'gs[0:2,3]',
    'gs[0:2,4]',
    'gs[0:2,5]',
    'gs[0:2,6]',
]

toff = 120
loff = 50 # 200 # 50 # 50 for 1° data, 200 for 0.25° data
time_months = pd.to_datetime(heat_ds.time.values)
longitudes = heat_ds.longitude.values

fig, axes = plt.subplots(1, 1, figsize=(12,3))
fig.suptitle(f'OHC Hovmoller (2°S-2°N)', fontsize=16, y=1.07)

axes.axis('off')

# initialize grid spec
gs = GridSpec(2, 7, figure=fig)
axs = []

for i,lt in enumerate(lead_times):
    ax = fig.add_subplot(eval(grid_list_[i]))
    im = ax.contourf(heat_ds.isel(prediction_timedelta=i), cmap=cmap2, norm=norm2, levels=levels2)
    title = 'Ground Truth' if i == 0 else f't+{lt}'
    ax.set_title(title, fontsize=14)
    axs.append(ax)

    if i in [0]:
        ax.set_ylabel('Time')
        ax.set_yticks([i for i in range(0,len(time_months),toff)])
        ax.set_yticklabels([f'{time_months[i].year}-{time_months[i].month:02d}' for i in range(0,len(time_months),toff)])
    else:
        ax.set_yticks([])
        ax.set_yticklabels([])

    ax.set_xlabel('Longitude')
    ax.set_xticks([i for i in range(0,len(longitudes),loff)])
    ax.set_xticklabels([int(longitudes[i]) for i in range(0,len(longitudes),loff)])


# single colorbar
cax = fig.add_axes([0.25, -0.05, 0.5, 0.03]) # [left, bottom, width, height]
cbar = fig.colorbar(
    im, ax=axs, cax=cax, 
    orientation='horizontal', pad=0.05, 
    aspect=60, shrink=0.9, label='J/K', 
    format=ScalarFormatter(), 
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.15, top=0.9, wspace=0.2, hspace=0.5
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'ocean_heat_content_hovmoller_high_time_res.png')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()
plt.close(fig)